In [1]:
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path="cracksense_MobileNetV2.tflite")
interpreter.allocate_tensors()

print(interpreter.get_output_details())

[{'name': 'StatefulPartitionedCall:0', 'index': 183, 'shape': array([1, 3]), 'shape_signature': array([-1,  3]), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


d:\anaconda\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [4]:
import tensorflow as tf
import numpy as np
from PIL import Image

# 1. Muat model TFLite
interpreter = tf.lite.Interpreter(model_path="cracksense_MobileNetV2.tflite")
interpreter.allocate_tensors()

# Dapatkan detail input dan output dari model
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_shape = input_details[0]['shape']
tinggi_gambar = input_shape[1]
lebar_gambar = input_shape[2]
print(f"Info: Model membutuhkan ukuran input {tinggi_gambar}x{lebar_gambar}\n")

# 2. Siapkan daftar 3 gambar yang ingin diuji
# Ganti dengan nama file gambar aslimu!
daftar_gambar = ['diagonal.jpg', 'vertikal.jpg', 'horizontal.jpg'] 
class_names = ['Diagonal', 'Horizontal', 'Vertikal']

print("="*50)
print("MULAI PROSES PREDIKSI 3 GAMBAR")
print("="*50)

# 3. Looping untuk memproses gambar satu per satu
for nama_gambar in daftar_gambar:
    print(f"\n--- Memproses: {nama_gambar} ---")
    try:
        # Muat dan Siapkan Gambar
        img = Image.open(nama_gambar).convert('RGB')
        img = img.resize((lebar_gambar, tinggi_gambar))
        
        img_array = np.array(img)
        img_array = np.expand_dims(img_array, axis=0) # Ubah ke bentuk [1, 224, 224, 3]
        img_array = np.float32(img_array) / 255.0     # Normalisasi

        # Masukkan array gambar ke dalam model
        interpreter.set_tensor(input_details[0]['index'], img_array)

        # Jalankan prediksi
        interpreter.invoke()

        # Ambil hasil prediksi (Output Array)
        output_data = interpreter.get_tensor(output_details[0]['index'])

        print(f"Output mentah: {output_data}")
        
        # Tampilkan persentase tiap kelas
        for i in range(len(class_names)):
            persentase = output_data[0][i] * 100
            print(f"  > {class_names[i]}: {persentase:.2f}%")

        # Cari kesimpulan akhir (menggunakan argmax)
        index_tertinggi = np.argmax(output_data)
        tebakan_kelas = class_names[index_tertinggi]
        keyakinan = output_data[0][index_tertinggi] * 100

        print(f"KESIMPULAN:")
        print(f"Model menebak '{nama_gambar}' adalah '{tebakan_kelas}' ({keyakinan:.2f}%)")
        print("-" * 50)
        
    except FileNotFoundError:
        print(f"ðŸš¨ Error: File '{nama_gambar}' tidak ditemukan!")
        print("-" * 50)

Info: Model membutuhkan ukuran input 224x224

MULAI PROSES PREDIKSI 3 GAMBAR

--- Memproses: diagonal.jpg ---
Output mentah: [[9.9840957e-01 6.9643639e-04 8.9413533e-04]]
  > Diagonal: 99.84%
  > Horizontal: 0.07%
  > Vertikal: 0.09%
KESIMPULAN:
Model menebak 'diagonal.jpg' adalah 'Diagonal' (99.84%)
--------------------------------------------------

--- Memproses: vertikal.jpg ---
Output mentah: [[2.6940415e-03 1.6635034e-04 9.9713969e-01]]
  > Diagonal: 0.27%
  > Horizontal: 0.02%
  > Vertikal: 99.71%
KESIMPULAN:
Model menebak 'vertikal.jpg' adalah 'Vertikal' (99.71%)
--------------------------------------------------

--- Memproses: horizontal.jpg ---
Output mentah: [[0.4808849  0.5112774  0.00783767]]
  > Diagonal: 48.09%
  > Horizontal: 51.13%
  > Vertikal: 0.78%
KESIMPULAN:
Model menebak 'horizontal.jpg' adalah 'Horizontal' (51.13%)
--------------------------------------------------
